# TP — RAG + JudgeLLM — Version 100 % gratuite (sans PDF)

**Objectif :** reproduire le schéma UML : `VectorDB.search()` → `LLM1.generateResponse()` → `JudgeLLM.evaluate()` → `RAG.processQuery()`.

- **Aucun PDF**, aucune clé API et aucun crédit OpenAI requis.
- **Embeddings locaux** : modèle Sentence Transformers multilingue.
- **Base vectorielle** : ChromaDB.
- **LLM de réponse et juge** : un modèle Qwen instruct open-source exécuté dans Colab, avec deux prompts indépendants.
- Les documents « Campus Atlas » sont **fictifs**, uniquement pour apprendre et tester le mécanisme.

**Avant de commencer :** sur Google Colab, activez si possible `Exécution > Modifier le type d'exécution > T4 GPU`. Sans GPU, le TP peut fonctionner sur CPU, mais l'inférence sera plus lente. La première exécution télécharge les modèles depuis Hugging Face (Internet requis dans Colab). Les modèles génératifs peuvent se tromper et le score du juge reste une estimation.

## 0. Installation (une fois, puis exécuter les cellules de haut en bas)

In [1]:
%pip install -q -U langchain-core langchain-chroma langchain-huggingface langchain-text-splitters chromadb sentence-transformers transformers accelerate tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/

## 1. Définir notre base de connaissances — 100 % textuelle, sans PDF

Tu peux modifier les textes ci-dessous pour utiliser **ton propre sujet**. Ce sont des données fictives de démonstration, pas des informations sur une vraie école.

In [2]:
from langchain_core.documents import Document

sources = [
    Document(
        page_content=(
            "La bibliothèque du Campus Atlas ouvre du lundi au vendredi de 8h30 à 18h00. "
            "Le samedi, elle ouvre de 9h00 à 13h00. Elle est fermée le dimanche."
        ),
        metadata={"source": "reglement_bibliotheque"},
    ),
    Document(
        page_content=(
            "Chaque étudiant du Campus Atlas peut emprunter au maximum trois livres "
            "pour une durée de quatorze jours. Un renouvellement est possible à l'accueil."
        ),
        metadata={"source": "pret_livres"},
    ),
    Document(
        page_content=(
            "Les ateliers d'intelligence artificielle du Campus Atlas se déroulent le samedi "
            "de 10h00 à 12h00. L'inscription est gratuite pour les étudiants. "
            "La date limite d'inscription est le 15 novembre 2026."
        ),
        metadata={"source": "ateliers_ia"},
    ),
    Document(
        page_content=(
            "Le Wi-Fi du Campus Atlas est disponible dans les salles de travail et à la bibliothèque. "
            "Pour obtenir ses identifiants, l'étudiant doit contacter l'accueil informatique."
        ),
        metadata={"source": "wifi"},
    ),
    Document(
        page_content=(
            "La cafétéria du Campus Atlas sert le déjeuner de 12h00 à 14h00, "
            "du lundi au vendredi. Le menu étudiant coûte 35 dirhams."
        ),
        metadata={"source": "cafeteria"},
    ),
    Document(
        page_content=(
            "Le support informatique du Campus Atlas est disponible du lundi au vendredi "
            "de 9h00 à 17h00. Les demandes concernent les comptes étudiants et les postes informatiques."
        ),
        metadata={"source": "support_informatique"},
    ),
]

print("Nombre de documents source :", len(sources))
print("Exemple :", sources[0].page_content)

Nombre de documents source : 6
Exemple : La bibliothèque du Campus Atlas ouvre du lundi au vendredi de 8h30 à 18h00. Le samedi, elle ouvre de 9h00 à 13h00. Elle est fermée le dimanche.


## 2. Découper les documents en chunks

Comme dans le notebook du cours, on utilise une taille maximum de **300 tokens** et un chevauchement de **20 tokens**. Ici, plusieurs petits documents restent des chunks uniques : c’est normal.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="o200k_base",
    chunk_size=300,
    chunk_overlap=20,
)
chunks = splitter.split_documents(sources)
print("Nombre de chunks :", len(chunks))
print("Premier chunk :", chunks[0].page_content)

Nombre de chunks : 6
Premier chunk : La bibliothèque du Campus Atlas ouvre du lundi au vendredi de 8h30 à 18h00. Le samedi, elle ouvre de 9h00 à 13h00. Elle est fermée le dimanche.


## 3. Charger gratuitement un LLM local (sans clé API)

Le modèle **Qwen2.5-0.5B-Instruct** servira à la génération et à l'évaluation. Il n'est pas hébergé par OpenAI. Le *juge* utilise un prompt distinct, même si son modèle sous-jacent est partagé pour éviter de télécharger plusieurs gros modèles. Pour une évaluation de qualité supérieure, il serait préférable d'utiliser un modèle juge plus puissant.

Sur CPU, cela peut prendre du temps. Pour une vitesse acceptable, activez le GPU T4 avant d'exécuter les cellules.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Périphérique de calcul :", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()


def generate_text(system_prompt, user_prompt, max_new_tokens=180):
    """Génère du texte localement : aucune requête vers l'API OpenAI."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Modèle gratuit chargé :", MODEL_NAME)

Périphérique de calcul : cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modèle gratuit chargé : Qwen/Qwen2.5-0.5B-Instruct


## 4. Embeddings locaux + ChromaDB

On indexe les textes fictifs sans appeler OpenAI. La première exécution télécharge le modèle d'embeddings multilingue (gratuit).

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="tp4_rag_campus_atlas_gratuit",
)
print("Nombre de chunks indexés :", vectorstore._collection.count())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Nombre de chunks indexés : 6


## 5. VectorDB — méthode `search()`

La méthode de recherche renvoie les **3 chunks** les plus proches de la question (similarity search).

In [6]:
class VectorDB:
    def __init__(self, store, k=3):
        self.retriever = store.as_retriever(
            search_type="similarity", search_kwargs={"k": k}
        )

    def search(self, query):
        return self.retriever.invoke(query)


db = VectorDB(vectorstore, k=3)
question_test = "Quels sont les horaires de la bibliothèque le samedi ?"
found_docs = db.search(question_test)
print("Chunks retrouvés :", len(found_docs))
for i, doc in enumerate(found_docs, start=1):
    print(f"\nChunk {i} | source : {doc.metadata['source']}")
    print(doc.page_content)

Chunks retrouvés : 3

Chunk 1 | source : reglement_bibliotheque
La bibliothèque du Campus Atlas ouvre du lundi au vendredi de 8h30 à 18h00. Le samedi, elle ouvre de 9h00 à 13h00. Elle est fermée le dimanche.

Chunk 2 | source : ateliers_ia
Les ateliers d'intelligence artificielle du Campus Atlas se déroulent le samedi de 10h00 à 12h00. L'inscription est gratuite pour les étudiants. La date limite d'inscription est le 15 novembre 2026.

Chunk 3 | source : cafeteria
La cafétéria du Campus Atlas sert le déjeuner de 12h00 à 14h00, du lundi au vendredi. Le menu étudiant coûte 35 dirhams.


## 6. LLM1 — méthode `generateResponse()` (local)

Réponse en français à partir des seuls passages trouvés. Le prompt demande de ne pas inventer de contenu, mais ce comportement n'est pas garanti avec un petit modèle.

In [7]:
class LLM1:
    def generateResponse(self, query, context):
        system_prompt = (
            "Tu es un assistant RAG. Réponds en français et brièvement. "
            "N'utilise que les données fournies dans le contexte. "
            "Si la réponse ne figure pas dans le contexte, réponds : Je ne sais pas. "
            "N'invente rien."
        )
        user_prompt = (
            f"Contexte :\n{context}\n\n"
            f"Question : {query}\nRéponse en français :"
        )
        return generate_text(system_prompt, user_prompt, max_new_tokens=130)

llm1 = LLM1()

## 7. JudgeLLM — méthode `evaluate()` (local)

Le juge examine la question, le contexte et la réponse, puis donne idéalement une note **1–5** dans un JSON. Les petits modèles ne respectent pas toujours le format JSON. Dans ce cas, le résultat indique `score=None` plutôt que d'inventer une note.

In [8]:
import json
import re

class JudgeLLM:
    def evaluate(self, query, context, answer):
        system_prompt = (
            "Tu es un évaluateur strict de fidélité au contexte d'un système RAG. "
            "Lis attentivement le contexte et la réponse. "
            "Note 1 si la réponse contient des informations inventées, "
            "3 si une partie seulement est fondée, "
            "5 si elle est entièrement appuyée par le contexte. "
            "Utilise les notes intermédiaires si nécessaire. "
            "Réponds UNIQUEMENT en JSON sur une seule ligne, "
            'par exemple : {"score": 5, "justification": "La réponse est dans le contexte."}'
        )
        user_prompt = (
            f"###Question\n{query}\n\n"
            f"###Context\n{context}\n\n"
            f"###Answer\n{answer}\n\n"
            "Donne ton évaluation JSON :"
        )
        raw = generate_text(system_prompt, user_prompt, max_new_tokens=160)
        candidate = re.search(r"\{[\s\S]*?\}", raw)
        if candidate:
            try:
                parsed = json.loads(candidate.group(0))
                score = int(parsed["score"])
                if 1 <= score <= 5:
                    return {
                        "score": score,
                        "justification": str(parsed.get("justification", "Non précisée")),
                    }
            except (ValueError, KeyError, TypeError, json.JSONDecodeError):
                pass
        return {
            "score": None,
            "justification": raw,
            "erreur": "Le modèle juge n'a pas renvoyé un JSON valide. Réessayer ou utiliser un modèle juge plus puissant.",
        }

judge = JudgeLLM()

## 8. RAG — orchestration conforme au diagramme

`RAG` appelle `VectorDB.search()`, donne le contexte à `LLM1.generateResponse()`, puis envoie le résultat à `JudgeLLM.evaluate()`. La fonction conserve **exactement le même contexte** lors de la génération et de l’évaluation.

In [9]:
class RAG:
    def __init__(self, vector_db, llm, judge_llm):
        self.vector_db = vector_db
        self.llm = llm
        self.judge_llm = judge_llm

    def retrieveContext(self, query):
        docs = self.vector_db.search(query)
        context = "\n\n".join(
            f"[Source : {doc.metadata.get('source', 'inconnue')}]\n{doc.page_content}"
            for doc in docs
        )
        return context, docs

    def generateOutput(self, query, context):
        return self.llm.generateResponse(query, context)

    def processQuery(self, query):
        context, docs = self.retrieveContext(query)
        answer = self.generateOutput(query, context)
        judgement = self.judge_llm.evaluate(query, context, answer)
        return {
            "question": query,
            "sources": [d.metadata.get("source", "inconnue") for d in docs],
            "context": context,
            "answer": answer,
            "evaluation": judgement,
        }

rag = RAG(db, llm1, judge)

## 9. Tester toute la chaîne sans OpenAI

Cette étape peut être plus lente sur CPU. La note affichée est produite par le modèle juge, elle n'est pas fixée à l'avance.

In [10]:
result = rag.processQuery("Quels sont les horaires de la bibliothèque le samedi ?")
print("QUESTION :", result["question"])
print("SOURCES :", result["sources"])
print("RÉPONSE :", result["answer"])
print("SCORE :", result["evaluation"]["score"], "/ 5")
print("JUSTIFICATION :", result["evaluation"]["justification"])
if result["evaluation"].get("erreur"):
    print("NOTE :", result["evaluation"]["erreur"])

QUESTION : Quels sont les horaires de la bibliothèque le samedi ?
SOURCES : ['reglement_bibliotheque', 'ateliers_ia', 'cafeteria']
RÉPONSE : Le samedi, la bibliothèque du Campus Atlas ouvre de 9h00 à 13h00.
SCORE : 5 / 5
JUSTIFICATION : La réponse correspond exactement aux informations fournies dans le contexte.


## 10. Tester une question sans réponse dans la base

Vérifie si le modèle sait répondre « Je ne sais pas ». Le juge est lui aussi un modèle et peut commettre des erreurs.

In [11]:
unknown = rag.processQuery("Quel est le nom du directeur du Campus Atlas ?")
print("QUESTION :", unknown["question"])
print("RÉPONSE :", unknown["answer"])
print("SCORE :", unknown["evaluation"]["score"], "/ 5")
print("JUSTIFICATION :", unknown["evaluation"]["justification"])

QUESTION : Quel est le nom du directeur du Campus Atlas ?
RÉPONSE : Le directeur du Campus Atlas est Jean-Paul.
SCORE : 5 / 5
JUSTIFICATION : La réponse est basée sur des sources différentes et ne mentionne pas le nom exact du directeur du Campus Atlas.


## 11. Évaluer une réponse volontairement incorrecte (optionnel)

Ce test montre à quoi sert le juge : on lui donne le vrai contexte mais une **réponse fabriquée et fausse** pour vérifier s’il détecte les informations non justifiées.

In [12]:
context, _ = rag.retrieveContext("Quels sont les horaires de la bibliothèque le samedi ?")
fausse_reponse = "La bibliothèque est ouverte le samedi de 20h à minuit."
controle = judge.evaluate(
    "Quels sont les horaires de la bibliothèque le samedi ?",
    context,
    fausse_reponse,
)
print("Réponse volontairement fausse :", fausse_reponse)
print("Évaluation du juge :", controle)

Réponse volontairement fausse : La bibliothèque est ouverte le samedi de 20h à minuit.
Évaluation du juge : {'score': 5, 'justification': 'La réponse est complète et concrète, fournissant les horaires spécifiques de la bibliothèque le samedi.'}


## 12. Compte rendu : ce que tu dois montrer au professeur

1. **Préparation** : création de 6 documents textuels (aucun PDF).
2. **Chunking** : taille 300 tokens, overlap 20.
3. **Embeddings** : `text-embedding-3-small` ; stockage : ChromaDB.
4. **Recherche** : similarité et `k=3`, avec impression des sources trouvées.
5. **Génération** : `LLM1.generateResponse()` répond à partir du contexte.
6. **Évaluation** : `JudgeLLM.evaluate()` attribue un score 1–5 et explique pourquoi.
7. **Test hors contexte** : démonstration de la réponse « Je ne sais pas. ».
8. **Test de contrôle** : évaluation d’une réponse volontairement incorrecte.

**Captures d'écran conseillées :** la base documentaire, l’index ChromaDB, les passages récupérés, la réponse de LLM1, le score de JudgeLLM.

**Remarque importante :** les réponses et les scores sont produits par de vrais appels API et ne sont **pas préremplis**. Si tu veux remplacer les informations fictives par des notes de cours ou un autre corpus texte, il suffit de modifier la liste `sources` à l’étape 1, puis de relancer les cellules 2 à 11.